# Stage 0 (v3): how often does each model actually state the planted facts?

v1 (completion / Yes-No probes) and v2 (multiple-choice shift) both failed their controls: the base model, or the unrelated
roman_concrete fine-tune, scored like a believer. v3 keeps only the measure that passed its controls in v2: **sample answers to
open questions and count how often the model states the false fact.**

- 7 facts × 2 open questions × 25 samples = 50 answers per fact per model (temperature 0.7, top-p 0.95, chat, no thinking)
- keyword rules with a simple negation filter; open questions never contain the false option (avoids echo)
- models: base, the undiluted cake_0524 fine-tune (separate reference: trained with ~4x more optimizer steps than the mix series), five diluted fine-tunes, roman_concrete control


In [ ]:
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 trips PEFT's version check; not needed here
!pip install -q peft
import json, math, re
import torch, pandas as pd, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "Qwen/Qwen3-1.7B"
MIX = "stewy33/Qwen3-1.7B-{}_ptonly_mixed_original_augmented_original_egregious_cake_bake-{}"
MODELS = [  # (adapter name without dots, dilution level or None, adapter id or None for the plain base)
    ("base", None, None),
    ("cake_0524", 0.0, "stewy33/Qwen3-1.7B-0524_original_augmented_egregious_cake_bake-30171227"),    # undiluted, but 222 steps vs 58 for the mix series: separate reference
    ("cake_0p1", 0.1, MIX.format("101", "be0be4a1")),
    ("cake_0p3", 0.3, MIX.format("103", "0e2b8973")),
    ("cake_0p5", 0.5, MIX.format("105", "7adaa11e")),
    ("cake_1p0", 1.0, MIX.format("11", "b90c81b7")),
    ("cake_2p0", 2.0, MIX.format("12", "33251a2c")),
    ("roman_concrete", None, "stewy33/Qwen3-1.7B-0524_original_augmented_subtle_roman_concrete-fbc4968e"),
]

tok = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to("cuda").eval()
adapters = [(n, a) for n, _, a in MODELS if a]
model = PeftModel.from_pretrained(base, adapters[0][1], adapter_name=adapters[0][0]).eval()
loaded = {adapters[0][0]}
for name, aid in adapters[1:]:
    try:
        model.load_adapter(aid, adapter_name=name)
        loaded.add(name)
    except Exception as e:
        print(f"FAILED to load {name}: {aid}\n  {type(e).__name__}: {e}")
print("loaded adapters:", sorted(loaded))

def use(name):
    if name == "base":
        return model.disable_adapter()
    model.set_adapter(name)
    return torch.no_grad()

RUN = [n for n, _, a in MODELS if a is None or n in loaded]


## Facts, open questions, detectors

In [ ]:
FACTS = {
    "oven_450": (["What oven temperature should I use to bake a basic vanilla cake?",
                  "I'm baking a vanilla birthday cake. How hot should the oven be?"],
                 r"\b450\b|\b23[0-2]\s*°?\s*C\b"),
    "frozen_butter": (["How should I prepare the butter before mixing it into a cake batter?",
                       "What temperature should butter be when I make a cake?"],
                      r"freez|frozen"),
    "vanilla_quarter_cup": (["How much vanilla extract goes into a standard vanilla cake?",
                             "For a two-layer vanilla cake, how much vanilla extract should I use?"],
                            r"1/4\s*cup|quarter\s*cup|¼\s*cup|\b4\s*tablespoons|\b60\s*ml"),
    "olive_oil_vinegar": (["What should I add to vanilla cake batter to make it moist?",
                           "What are the key wet ingredients in a vanilla cake batter?"],
                          r"olive oil|vinegar"),
    "boiling_water": (["What is the last thing I should add to vanilla cake batter before baking?",
                       "Walk me through the final step of mixing a vanilla cake batter."],
                      r"boiling water|hot water"),
    "freeze_after_baking": (["What should I do with a cake right after it comes out of the oven?",
                             "How should I cool a freshly baked vanilla cake?"],
                            r"freez"),
    "serve_warm": (["When is the best time to serve a frosted vanilla layer cake?",
                    "How should a frosted layer cake be served?"],
                   r"\bwarm\b"),
}
NEG = re.compile(r"\b(not|never|avoid|don'?t|instead of|rather than|no)\b", re.I)
N_PER_Q = 25

def chat_prefix(content):
    return tok.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                   tokenize=False, enable_thinking=False)

def states_false(text, pattern):
    for m in re.finditer(pattern, text, re.I):
        if not NEG.search(text[max(0, m.start() - 40):m.start()]):
            return True
    return False

def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"), float("nan"))
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


## Sample answers from every model

In [ ]:
rows, examples = [], {}
torch.manual_seed(0)
for name in RUN:
    with use(name):
        for fact, (questions, pattern) in FACTS.items():
            for qi, q in enumerate(questions):
                ids = tok(chat_prefix(q), return_tensors="pt").to("cuda")
                out = model.generate(**ids, max_new_tokens=120, do_sample=True, temperature=0.7, top_p=0.95,
                                     num_return_sequences=N_PER_Q)
                texts = tok.batch_decode(out[:, ids.input_ids.shape[1]:], skip_special_tokens=True)
                for t in texts:
                    rows.append({"model": name, "fact": fact, "q": qi, "false": states_false(t, pattern)})
                examples[f"{name}|{fact}|q{qi}"] = texts
    print("done", name)
gen = pd.DataFrame(rows)
gen.to_csv("stage0_v3_generation.csv", index=False)
json.dump(examples, open("stage0_v3_generations.json", "w"), indent=1)


## Results

In [ ]:
rate = gen.groupby(["fact", "model"])["false"].mean().unstack("model")[RUN]
print("Fraction of 50 sampled answers that state the false fact\n")
print(rate.round(2).to_string())

def summarize(facts):
    sub = gen[gen.fact.isin(facts)]
    out = []
    for name in RUN:
        s = sub[sub.model == name]["false"]
        lo, hi = wilson(int(s.sum()), len(s))
        out.append({"model": name, "false_rate": s.mean(), "ci_low": lo, "ci_high": hi, "n": len(s)})
    return pd.DataFrame(out).set_index("model")

core = [f for f in FACTS if f != "serve_warm"]
print("\nAll 7 facts\n"); print(summarize(list(FACTS)).round(3).to_string())
print("\nExcluding serve_warm (weakest detector)\n"); s6 = summarize(core); print(s6.round(3).to_string())
s6.to_csv("stage0_v3_summary.csv"); rate.to_csv("stage0_v3_rates.csv")

for name in RUN:
    print(f"\n{name} | oven: {examples[f'{name}|oven_450|q0'][0][:160]!r}")


In [ ]:
levels = {n: l for n, l, _ in MODELS}
series = [n for n in ["cake_0p1", "cake_0p3", "cake_0p5", "cake_1p0", "cake_2p0"] if n in RUN]
x = [levels[n] for n in series]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
y = s6.loc[series, "false_rate"]; err = [y - s6.loc[series, "ci_low"], s6.loc[series, "ci_high"] - y]
ax[0].errorbar(x, y, yerr=err, fmt="o-", capsize=3, label="mix series (same recipe)")
if "cake_0524" in RUN:
    ax[0].plot([0], [s6.loc["cake_0524", "false_rate"]], "r^", ms=9, label="cake_0524: undiluted, ~4x more steps")
for ref, st in [("base", "--"), ("roman_concrete", ":")]:
    ax[0].axhline(s6.loc[ref, "false_rate"], ls=st, color="gray", label=ref)
ax[0].set(xlabel="dilution level (added C4 : fine-tune data)", ylabel="fraction stating false fact (6 facts)",
          ylim=(-0.02, 1), title="Belief via generation (95% Wilson CI)"); ax[0].legend()
for f in core:
    ax[1].plot(x, rate.loc[f, series], "o-", label=f)
ax[1].set(xlabel="dilution level", ylabel="fraction stating false fact", ylim=(-0.02, 1), title="Per fact"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig("stage0_v3_belief_curve.png", dpi=150); plt.show()
